# Unit 4: PyTorch RNN 模块

## 学习目标
- 掌握 `nn.RNN`、`nn.LSTM`、`nn.GRU` 的 API
- 理解输入/输出形状约定 (batch_first)
- 掌握 hidden state 和 cell state 的机制
- 完成一个时间序列预测实战

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 4.1 PyTorch RNN 总览

| 模块 | 全称 | 门控机制 | 适用场景 |
|------|------|----------|----------|
| `nn.RNN` | Vanilla RNN | 无 | 简单序列，短依赖 |
| `nn.LSTM` | Long Short-Term Memory | 3门 (遗忘/输入/输出) | 长序列，长期依赖 |
| `nn.GRU` | Gated Recurrent Unit | 2门 (重置/更新) | 类似 LSTM，计算更快 |

**统一的输入形状**（`batch_first=True`）：
- 输入: `(batch_size, seq_len, input_size)`
- 输出: `(batch_size, seq_len, hidden_size)`
- 隐藏状态: `(num_layers, batch_size, hidden_size)`

## 4.2 nn.RNN

最简单的 RNN 实现。适合教学理解，实际项目中通常用 LSTM/GRU。  
所以严格来说，RNN 前向传播唯一必须的输入只有 x。h0 之所以可以单独传入，是因为在某些场景下你需要自定义初始状态（如续写文本、迁移隐藏状态等），但它本质上仍然属于"数据"而非"模型参数"。不传 h0，PyTorch 默认使用全零张量作为初始隐藏状态。   
h0 = torch.zeros(2, batch_size, hidden_size)  # (num_layers, batch, hidden)  
h_n 的形状 (num_layers, batch, hidden_size) 是由 RNN 的数学递推本质决定的，它不受 batch_first=True 的影响。
| 张量 | batch_first=False (默认) | batch_first=True |
| :--- | :--- | :--- |
| x | (seq, batch, feature) | (batch, seq, feature) ✅ 变了 |
| output | (seq, batch, hidden) | (batch, seq, hidden) ✅ 变了 |
| `h0` / `h_n` | `(num_layers, batch, hidden)` | `(num_layers, batch, hidden)` ❌ 没变 |



```text
              t=1    t=2    t=3   ...   t=T
            ┌──────┬──────┬──────┬─────┬──────┐
  Layer 1   │ H¹₁  │ H¹₂  │ H¹₃  │ ... │ H¹_T │
            ├──────┼──────┼──────┼─────┼──────┤
  Layer 2   │ H²₁  │ H²₂  │ H²₃  │ ... │ H²_T │
            ├──────┼──────┼──────┼─────┼──────┤
  ...       │      │      │      │     │      │
            ├──────┼──────┼──────┼─────┼──────┤
  Layer L   │ Hᴸ₁  │ Hᴸ₂  │ Hᴸ₃  │ ... │ Hᴸ_T │ ← output (最后一行)
            └──────┴──────┴──────┴─────┴──────┘
                                            ↑
                                        h_n (最后一列)
```

In [ ]:
# nn.RNN 基本用法
batch_size, seq_len, input_size, hidden_size = 4, 10, 8, 16

rnn = nn.RNN(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=2,            # 2层堆叠
    batch_first=True,        # 输入形状: (batch, seq, feature)
    dropout=0.0,             # 层间 dropout (num_layers>1 时有效)
    bidirectional=False,     # 是否双向
)

x = torch.randn(batch_size, seq_len, input_size)
h0 = torch.zeros(2, batch_size, hidden_size)  # (num_layers, batch, hidden)

# 前向传播
output, h_n = rnn(x, h0)
# output: (batch, seq_len, hidden_size):最后一层
# h_n: (num_layers, batch, hidden_size):最后一列

print(f"输入:    {x.shape}")
print(f"输出:    {output.shape}    # (batch, seq_len, hidden_size)")
print(f"最终 h_n: {h_n.shape}    # (num_layers, batch, hidden_size)")
print(f"\n每个时间步都有输出: output 包含所有 seq_len 步的输出")
print(f"output[:, -1, :] == h_n[-1]: {(output[:, -1, :] - h_n[-1]).abs().max().item():.10f}")

## 4.3 nn.LSTM

LSTM 引入了**门控机制**和**细胞状态 (cell state)** $c_t$：

- **遗忘门** $f_t$: 控制保留多少旧信息
- **输入门** $i_t$: 控制写入多少新信息
- **输出门** $o_t$: 控制输出多少信息
- **细胞状态** $c_t$: 长期记忆，梯度可以无损流动（缓解梯度消失）

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
$$\tilde{c}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$$
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(c_t)$$

In [ ]:
# nn.LSTM 基本用法
lstm = nn.LSTM(
    input_size=8,
    hidden_size=16,
    num_layers=2,
    batch_first=True,
    dropout=0.2,
)

x = torch.randn(4, 10, 8)
h0 = torch.zeros(2, 4, 16)  # 初始隐藏状态
c0 = torch.zeros(2, 4, 16)  # 初始细胞状态

output, (h_n, c_n) = lstm(x, (h0, c0))

print(f"输出 output:    {output.shape}  # 所有时间步的输出")
print(f"最终 h_n:        {h_n.shape}     # 最终隐藏状态")
print(f"最终 c_n:        {c_n.shape}     # 最终细胞状态")
print(f"\nLSTM 参数数量: {sum(p.numel() for p in lstm.parameters()):,}")

## 4.4 nn.GRU

GRU 是 LSTM 的简化版，将遗忘门和输入门合并为**更新门**：

- **重置门** $r_t$: 控制忽略多少旧状态
- **更新门** $z_t$: 控制保留多少旧状态 vs 接受多少新状态

$$z_t = \sigma(W_z \cdot [h_{t-1}, x_t])$$
$$r_t = \sigma(W_r \cdot [h_{t-1}, x_t])$$
$$\tilde{h}_t = \tanh(W \cdot [r_t \odot h_{t-1}, x_t])$$
$$h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

In [ ]:
gru = nn.GRU(
    input_size=8,
    hidden_size=16,
    num_layers=2,
    batch_first=True,
)

x = torch.randn(4, 10, 8)
h0 = torch.zeros(2, 4, 16)

output, h_n = gru(x, h0)
print(f"输出: {output.shape}")
print(f"最终 h_n: {h_n.shape}")
print(f"GRU 参数数量: {sum(p.numel() for p in gru.parameters()):,}")
print(f"对比 LSTM 参数: {sum(p.numel() for p in nn.LSTM(8,16,2).parameters()):,}")

## 4.5 Hidden State 机制详解

隐藏状态的初始化和传递是关键操作。

In [ ]:
# Hidden State 操作演示
lstm = nn.LSTM(4, 8, num_layers=2, batch_first=True).to(device)

# 方式1: 不传 hidden state (自动初始化为零)
x = torch.randn(3, 5, 4, device=device)
out1, _ = lstm(x)

# 方式2: 传入自定义初始 hidden state
h0 = torch.randn(2, 3, 8, device=device)
c0 = torch.randn(2, 3, 8, device=device)
out2, (hn, cn) = lstm(x, (h0, c0))

# 方式3: Stateful RNN — 保留上批次的 hidden state
# 将长序列分成多个 batch，hidden state 跨 batch 传递
h_state = torch.zeros(2, 3, 8, device=device)
c_state = torch.zeros(2, 3, 8, device=device)

chunk1 = x[:, :2, :]  # 前2步
chunk2 = x[:, 2:, :]  # 后3步

out_c1, (h_state, c_state) = lstm(chunk1, (h_state, c_state))
out_c2, (h_state, c_state) = lstm(chunk2, (h_state, c_state))

print(f"Chunk1 输出: {out_c1.shape}")
print(f"Chunk2 输出: {out_c2.shape}")
print(f"\nh_state.detach() 是断开跨 batch 计算图的关键")
print(f"通常: h_state = h_state.detach() 用于截断 BPTT")

## 4.6 时间序列预测实战

用 LSTM 预测航空乘客量（经典时间序列数据集）。

In [ ]:
# 生成合成时间序列数据（模拟航空乘客量）
def generate_time_series(n_points=300):
    t = np.arange(n_points)
    trend = 0.02 * t
    season1 = 10 * np.sin(2 * np.pi * t / 50)    # 长周期
    season2 = 5 * np.sin(2 * np.pi * t / 12)     # 短周期
    noise = 2 * np.random.randn(n_points)
    return trend + season1 + season2 + noise

data = generate_time_series(400)

plt.figure(figsize=(14, 4))
plt.plot(data, linewidth=1)
plt.title("合成时间序列")
plt.xlabel("Time")
plt.ylabel("Value")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 数据准备: 用历史 window_size 个点预测未来 forecast_size 个点
def prepare_sequences(data, window_size, forecast_size):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_size + 1):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size:i+window_size+forecast_size])
    return np.array(X), np.array(y)

window_size = 30
forecast_size = 10

# 标准化
data_mean = data.mean()
data_std = data.std()
data_norm = (data - data_mean) / data_std

X, y = prepare_sequences(data_norm, window_size, forecast_size)

# 分割训练/测试
split = int(0.8 * len(X))
X_train = torch.tensor(X[:split], dtype=torch.float32).unsqueeze(-1)
y_train = torch.tensor(y[:split], dtype=torch.float32)
X_test = torch.tensor(X[split:], dtype=torch.float32).unsqueeze(-1)
y_test = torch.tensor(y[split:], dtype=torch.float32)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

In [ ]:
# Seq2Seq LSTM 预测模型
class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, forecast_size, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, forecast_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # 取最后时间步的输出进行预测
        last_out = lstm_out[:, -1, :]  # (batch, hidden_size)
        return self.fc(last_out)        # (batch, forecast_size)

model = TimeSeriesLSTM(input_size=1, hidden_size=64, forecast_size=forecast_size).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

print(f"模型参数: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 训练
epochs = 300
train_losses = []

for epoch in tqdm(range(epochs)):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train.to(device))
    loss = criterion(pred, y_train.to(device))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    train_losses.append(loss.item())

print(f"Final Train Loss: {train_losses[-1]:.6f}")

In [ ]:
# 评估
model.eval()
with torch.no_grad():
    test_pred = model(X_test.to(device))
    test_loss = criterion(test_pred, y_test.to(device))
    print(f"Test Loss: {test_loss.item():.6f}")

# 可视化
test_pred_np = test_pred.cpu().numpy()
y_test_np = y_test.numpy()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 训练损失
axes[0].plot(train_losses[10:], linewidth=1)
axes[0].set_title("Training Loss (前10步省略)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].grid(True, alpha=0.3)

# 预测 vs 真实 (取第一个测试样本)
sample_idx = 0
axes[1].plot(range(window_size), X_test[sample_idx, :, 0], 'b-', label="Input", alpha=0.5)
axes[1].plot(range(window_size, window_size+forecast_size), y_test_np[sample_idx], 'g-o', label="True Future", markersize=5)
axes[1].plot(range(window_size, window_size+forecast_size), test_pred_np[sample_idx], 'r-s', label="Predicted", markersize=5)
axes[1].axvline(x=window_size-1, color='k', linestyle='--', alpha=0.5)
axes[1].set_title(f"预测样本 #{sample_idx}")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 要点总结

| 模块 | 输出 | 隐藏状态 | 关键特点 |
|------|------|----------|----------|
| nn.RNN | (output, h_n) | 只有 h | 简单，长期依赖差 |
| nn.LSTM | (output, (h_n, c_n)) | h + c | 门控，长期记忆好 |
| nn.GRU | (output, h_n) | 只有 h | 2门，效率高 |

**输入形状口诀** (batch_first=True):
- 输入: **(B, T, F)** — Batch × Time × Feature
- 输出: **(B, T, H)** — 包含所有时间步
- 隐状态: **(L, B, H)** — Layers × Batch × Hidden

### 练习
1. 替换 LSTM 为 GRU，对比训练速度和效果
2. 改为多步预测（每步输出一个值），而非一次性预测整个序列
3. 尝试 `bidirectional=True`，观察对时间序列预测的影响